# IDRA Data Science & AI Capstone Project
## Retail Intelligence: Analysing Sales Drivers and Predicting Product Demand

**Selected Project:** Project 6 — Retail Intelligence  
**Dataset:** Walmart Weekly Sales Dataset  
**Target Variable:** `Weekly_Sales`  
**Problem Type:** Regression

This notebook is the reproducible computational record for the capstone. It follows the IDRA sequence:
**loading → understanding → cleaning → preprocessing → EDA → statistical analysis → feature engineering → modelling → evaluation → interpretation**.


## Project Details

- **Student Name:** `Varun Kumar`
- **Institute:** `Guru Nanak Dev University`
- **Roll Number:** `28152302128`
- **IDRA Enrollment Number:** `IDRA-2026-167229`
- **Programme:** Data Science & AI


# 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

sns.set_theme(style="whitegrid")

# 2. Load Dataset


In [ ]:
# Keep the CSV in the same folder as this notebook, or change this path.
FILE_PATH = "datasets list/P_6_Walmart.csv"

df = pd.read_csv(FILE_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## 2.1 Initial Inspection


In [ ]:
display(df.head())
display(df.tail())
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T)

# 3. Dataset Understanding

The target is `Weekly_Sales`, which is continuous. Therefore the prediction problem is formulated as **regression**.


## 3.1 Data Dictionary

| Variable | Meaning | Type | Role |
|---|---|---|---|
| `Store` | Store identifier | Integer | Feature |
| `Date` | Week of observation | Date/string before conversion | Feature source |
| `Weekly_Sales` | Weekly sales amount | Continuous | **Target** |
| `Holiday_Flag` | Holiday-week indicator | Binary | Feature |
| `Temperature` | Temperature | Continuous | Feature |
| `Fuel_Price` | Fuel price | Continuous | Feature |
| `CPI` | Consumer Price Index | Continuous | Feature |
| `Unemployment` | Unemployment rate | Continuous | Feature |


# 4. Data Quality Assessment


## 4.1 Missing Values


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing_summary)


## 4.2 Duplicate Records


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


## 4.3 Unique Values / Category Checks


In [ ]:
for col in df.columns:
    print(f"\n{col} | unique values = {df[col].nunique()}")
    if df[col].nunique() <= 20:
        print(df[col].unique())


## 4.4 Numerical Validity Checks


In [ ]:
numeric_columns = [
    "Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"
]

validity = pd.DataFrame({
    "Negative_Count": [(df[c] < 0).sum() for c in numeric_columns],
    "Zero_Count": [(df[c] == 0).sum() for c in numeric_columns]
}, index=numeric_columns)

display(validity)


# 5. Data Cleaning


In [ ]:
data = df.copy()

before = len(data)
data = data.drop_duplicates().copy()
after = len(data)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)


## 5.1 Date Conversion


In [ ]:
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True, errors="coerce")

print("Date dtype:", data["Date"].dtype)
print("Invalid dates:", data["Date"].isna().sum())
print("Date range:", data["Date"].min(), "to", data["Date"].max())


## 5.2 Cleaning Validation


In [ ]:
display(pd.DataFrame({
    "Missing_Count": data.isna().sum(),
    "Data_Type": data.dtypes.astype(str)
}))

print("Remaining duplicate rows:", data.duplicated().sum())


# 6. Exploratory Data Analysis (EDA)


## 6.1 Weekly Sales Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data["Weekly_Sales"], kde=True)
plt.title("Distribution of Weekly Sales")
plt.xlabel("Weekly Sales")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Skewness:", round(data["Weekly_Sales"].skew(), 3))
print("Kurtosis:", round(data["Weekly_Sales"].kurtosis(), 3))


## 6.2 Store-Level Analysis


In [ ]:
store_summary = (
    data.groupby("Store")["Weekly_Sales"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(store_summary.head(10))
display(store_summary.tail(10))


In [ ]:
plt.figure(figsize=(15, 6))
sns.barplot(data=data, x="Store", y="Weekly_Sales", estimator="mean", errorbar=None)
plt.title("Average Weekly Sales by Store")
plt.xlabel("Store")
plt.ylabel("Average Weekly Sales")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 6.3 Holiday Analysis


In [ ]:
holiday_summary = data.groupby("Holiday_Flag")["Weekly_Sales"].agg(
    ["count", "mean", "median", "std", "sum"]
)
display(holiday_summary)


In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=data, x="Holiday_Flag", y="Weekly_Sales")
plt.title("Weekly Sales: Holiday vs Non-Holiday")
plt.xlabel("Holiday Flag")
plt.ylabel("Weekly Sales")
plt.tight_layout()
plt.show()


## 6.4 Time-Based Analysis


In [ ]:
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Week"] = data["Date"].dt.isocalendar().week.astype(int)
data["Quarter"] = data["Date"].dt.quarter

print("Year range:", data["Year"].min(), "to", data["Year"].max())


In [ ]:
weekly_total = data.groupby("Date", as_index=False)["Weekly_Sales"].sum()

plt.figure(figsize=(15, 6))
plt.plot(weekly_total["Date"], weekly_total["Weekly_Sales"])
plt.title("Total Weekly Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Weekly Sales")
plt.tight_layout()
plt.show()


In [ ]:
yearly_sales = data.groupby("Year")["Weekly_Sales"].agg(["mean", "median", "sum"])
display(yearly_sales)


In [ ]:
monthly_sales = data.groupby("Month")["Weekly_Sales"].agg(["mean", "median", "sum"])
display(monthly_sales)

plt.figure(figsize=(10, 5))
sns.barplot(x=monthly_sales.index, y=monthly_sales["mean"].values, errorbar=None)
plt.title("Average Weekly Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Weekly Sales")
plt.tight_layout()
plt.show()


## 6.5 Feature vs Sales Relationships


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="Temperature", y="Weekly_Sales", alpha=0.5)
plt.title("Temperature vs Weekly Sales")
plt.xlabel("Temperature")
plt.ylabel("Weekly Sales")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="Fuel_Price", y="Weekly_Sales", alpha=0.5)
plt.title("Fuel Price vs Weekly Sales")
plt.xlabel("Fuel_Price")
plt.ylabel("Weekly Sales")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="CPI", y="Weekly_Sales", alpha=0.5)
plt.title("CPI vs Weekly Sales")
plt.xlabel("CPI")
plt.ylabel("Weekly Sales")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=data, x="Unemployment", y="Weekly_Sales", alpha=0.5)
plt.title("Unemployment vs Weekly Sales")
plt.xlabel("Unemployment")
plt.ylabel("Weekly Sales")
plt.tight_layout()
plt.show()


# 7. Statistical Analysis


In [ ]:
sales = data["Weekly_Sales"]

Q1 = sales.quantile(0.25)
Q3 = sales.quantile(0.75)
IQR = Q3 - Q1

stats_summary = pd.DataFrame({
    "Mean": [sales.mean()],
    "Median": [sales.median()],
    "Std_Dev": [sales.std()],
    "Variance": [sales.var()],
    "Q1": [Q1],
    "Q3": [Q3],
    "IQR": [IQR],
    "Minimum": [sales.min()],
    "Maximum": [sales.max()]
}).T

stats_summary.columns = ["Weekly_Sales"]
display(stats_summary)


## 7.1 Group-Level Statistics


In [ ]:
display(
    data.groupby("Holiday_Flag")["Weekly_Sales"].agg(
        Count="count",
        Mean="mean",
        Median="median",
        Std_Dev="std",
        Minimum="min",
        Maximum="max"
    )
)


# 8. Correlation Analysis


In [ ]:
corr_cols = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
correlation_matrix = data[corr_cols].corr()

display(correlation_matrix)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()


# 9. Outlier Analysis


In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[
    (sales < lower_bound) | (sales > upper_bound)
].copy()

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential Weekly_Sales outliers:", len(outliers))
print("Percentage:", round(len(outliers) / len(data) * 100, 2), "%")


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x=sales)
plt.title("Boxplot of Weekly Sales")
plt.xlabel("Weekly Sales")
plt.tight_layout()
plt.show()


# 10. Feature Engineering

The `Date` field is transformed into `Year`, `Month`, `Week`, and `Quarter`.
These variables provide the model with structured time information while avoiding direct use of the raw datetime value.


In [ ]:
display(data[["Date", "Year", "Month", "Week", "Quarter"]].head(10))


# 11. Feature and Target Preparation


In [ ]:
X = data.drop(columns=["Weekly_Sales", "Date"]).copy()
y = data["Weekly_Sales"].copy()

# Store is an identifier, so treat it as categorical rather than continuous.
X["Store"] = X["Store"].astype(str)

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())


## 11.1 Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


# 12. Preprocessing Pipeline


In [ ]:
categorical_features = ["Store", "Holiday_Flag", "Year", "Month", "Week", "Quarter"]
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_transformer, categorical_features),
    ("numeric", numeric_transformer, numeric_features)
])

print("Preprocessing pipeline ready.")


# 13. Model Development

The following regression models are compared:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    )
}

pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}


## 13.1 Train Models


In [ ]:
trained_models = {}

for name, pipeline in pipelines.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

print("Training complete.")


# 14. Model Evaluation


In [ ]:
evaluation_rows = []

for name, model in trained_models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    evaluation_rows.append({
        "Model": name,
        "Train_MAE": mean_absolute_error(y_train, train_pred),
        "Test_MAE": mean_absolute_error(y_test, test_pred),
        "Train_RMSE": np.sqrt(train_mse),
        "Test_RMSE": np.sqrt(test_mse),
        "Train_R2": r2_score(y_train, train_pred),
        "Test_R2": r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(evaluation_rows).sort_values("Test_RMSE")
display(results_df)


## 14.1 Best Model


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model by lowest Test RMSE:", best_model_name)


# 15. Overfitting / Underfitting Analysis


In [ ]:
overfit_check = results_df.copy()
overfit_check["R2_Gap"] = (
    overfit_check["Train_R2"] - overfit_check["Test_R2"]
)
overfit_check["RMSE_Ratio"] = (
    overfit_check["Test_RMSE"] / overfit_check["Train_RMSE"]
)

display(
    overfit_check[
        ["Model", "Train_R2", "Test_R2", "R2_Gap",
         "Train_RMSE", "Test_RMSE", "RMSE_Ratio"]
    ]
)


### Interpretation Guidance

Use the actual values above to determine whether each model shows:
- reasonable generalization,
- overfitting through a large train/test performance gap, or
- underfitting through weak performance on both training and test data.


# 16. Actual vs Predicted Weekly Sales


In [ ]:
best_predictions = best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_predictions, alpha=0.5)

min_value = min(y_test.min(), best_predictions.min())
max_value = max(y_test.max(), best_predictions.max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.title(f"Actual vs Predicted Weekly Sales — {best_model_name}")
plt.xlabel("Actual Weekly Sales")
plt.ylabel("Predicted Weekly Sales")
plt.tight_layout()
plt.show()


# 17. Residual Analysis


In [ ]:
residuals = y_test - best_predictions

display(pd.Series(residuals, name="Residual").describe())


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True)
plt.title(f"Residual Distribution — {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(best_predictions, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.title(f"Residuals vs Predicted Sales — {best_model_name}")
plt.xlabel("Predicted Weekly Sales")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()


# 18. Feature Importance / Coefficients


In [ ]:
final_estimator = best_model.named_steps["model"]
fitted_preprocessor = best_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(final_estimator, "feature_importances_"):
    values = final_estimator.feature_importances_
    feature_table = pd.DataFrame({
        "Feature": feature_names,
        "Importance": values
    }).sort_values("Importance", ascending=False)

    display(feature_table.head(20))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=feature_table.head(15), x="Importance", y="Feature")
    plt.title(f"Top Feature Importances — {best_model_name}")
    plt.tight_layout()
    plt.show()

elif hasattr(final_estimator, "coef_"):
    values = final_estimator.coef_
    feature_table = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": values,
        "Absolute_Coefficient": np.abs(values)
    }).sort_values("Absolute_Coefficient", ascending=False)

    display(feature_table.head(20))


# 19. Final Model Metrics


In [ ]:
final_mae = mean_absolute_error(y_test, best_predictions)
final_mse = mean_squared_error(y_test, best_predictions)
final_rmse = np.sqrt(final_mse)
final_r2 = r2_score(y_test, best_predictions)

final_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Value": [final_mae, final_mse, final_rmse, final_r2]
})

display(final_metrics)


# 20. Key Findings

After running the notebook, replace this section with the actual evidence-based findings.

### Finding 1 — Store-level variation
Identify the stores with the highest and lowest average weekly sales and cite the corresponding table/figure in the report.

### Finding 2 — Time-based patterns
Describe the meaningful yearly, monthly and/or weekly sales patterns observed.

### Finding 3 — Holiday behaviour
Compare holiday and non-holiday weeks using group-level statistics and the boxplot.

### Finding 4 — Sales-driver associations
Identify the strongest relationships with `Weekly_Sales` from the correlation analysis and scatterplots. Do not interpret correlation as causation.

### Finding 5 — Predictive performance
Report the selected model and its test MAE, RMSE and R². Discuss whether the training/testing gap suggests reasonable generalization.


# 21. Limitations

Complete this section using limitations that are actually supported by the project. Areas to assess include:

- whether important sales drivers are absent from the dataset
- the limited set of economic/environmental variables available
- representativeness of the observed period and stores
- correlation versus causation
- limitations of the selected model
- generalization to future or different store conditions


# 22. Conclusion

The final conclusion should return to the original prediction problem and objectives.

State:
1. what the EDA established,
2. which relationships were most important,
3. which model performed best,
4. how well it predicted weekly sales, and
5. what the results mean in practical terms.

Do not introduce new evidence here.


# 23. Recommendations and Future Work

Recommendations must follow:

**Finding → Evidence → Implication → Action**

Future work may consider additional business variables, alternative modelling approaches, stronger validation, or deployment-oriented improvements where justified by the findings.


# 24. Final Reproducibility Check


In [ ]:
print("Final dataset shape:", data.shape)
print("Selected model:", best_model_name)
print("Test MAE:", round(final_mae, 3))
print("Test RMSE:", round(final_rmse, 3))
print("Test R²:", round(final_r2, 4))
print("\nNotebook workflow completed successfully.")
